# KBO deep_learning_state — Colab runner

Thin wrapper. No training logic lives in this notebook; every cell shells out to
`src/deep_learning_state/`. Edit the code on the Mac, push, re-run here.

**Runtime > Change runtime type > GPU** before running anything.

## 1. Clone / pull

In [ ]:
REPO = 'https://github.com/Gromiit/kbo-control.git'
import os, pathlib
if pathlib.Path('/content/kbo/.git').exists():
    !cd /content/kbo && git pull --ff-only
else:
    !git clone $REPO /content/kbo
os.chdir('/content/kbo')
!git rev-parse --short HEAD


## 2. Dependencies + CUDA check

In [ ]:
!bash scripts/setup_colab.sh

## 3. Attach the shards

Shards are built on the Mac and are NOT in git.

**Copy them onto the local disk, do not symlink into Drive.** The dataset
memory-maps `.npy` files; mmap over the Drive FUSE mount degrades to a network
round trip per page and turns a 40-second epoch into tens of minutes. The
shards are ~1.3 GB, /content has ~70 GB.

Checkpoints and results DO go to Drive (small, and they must survive a
disconnect) via `KBO_CKPT` / `KBO_EXP`.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')

# shards: Drive -> local disk (one copy per runtime)
!mkdir -p /content/kbo/data/sequences
!rsync -a --info=progress2 /content/drive/MyDrive/kbo/sequences/ /content/kbo/data/sequences/
!du -sh /content/kbo/data/sequences && ls /content/kbo/data/sequences

# checkpoints + results: straight to Drive, so a disconnect costs nothing
import os
os.environ['KBO_CKPT'] = '/content/drive/MyDrive/kbo/out/checkpoints'
os.environ['KBO_EXP']  = '/content/drive/MyDrive/kbo/out'
!mkdir -p "$KBO_CKPT"


## 4. Smoke test (always first)

In [ ]:
!SMOKE_ONLY=1 bash scripts/train_colab.sh

## 5. Full training

Only after the smoke test passes **and** you have been told to run it.
Seeds run sequentially — one GPU, one model at a time.

In [ ]:
!CONFIG=configs/gru_full.yaml SEEDS='42' bash scripts/train_colab.sh

## 6. Results

`KBO_EXP` already points at Drive, so `results.csv`, the per-epoch traces and
the checkpoints are written there directly. Nothing to copy.


In [ ]:
!ls -la "$KBO_EXP"
!cat "$KBO_EXP/results.csv"
